<style>
:root { --dtin-ice: #DCEFF5; --dtin-navy: #15354A; --dtin-cyan: #2B7A9B; --dtin-gray: #5E6B73; }
.dtin-title { color: var(--dtin-navy); font: 650 2rem/1.15 system-ui, sans-serif; margin-bottom: .35rem; }
.dtin-lead { color: var(--dtin-gray); font: 1rem/1.55 system-ui, sans-serif; max-width: 72ch; }
.dtin-contract { background: var(--dtin-ice); border-left: 4px solid var(--dtin-cyan); padding: .75rem 1rem; max-width: 68ch; }
</style>

<div class="dtin-title">D-TIN surface reconstruction</div>
<p class="dtin-lead">Reconstruct an ice-sheet surface from track-sampled
altimetry with D-TIN. The paper's MS-D-TIN and GStatSim ordinary-kriging
methods are included for comparison.</p>

<div class="dtin-contract"><strong>Input CSV</strong><br>
Required columns: <code>lon</code> (degrees), <code>lat</code> (degrees), and
<code>elev</code> (metres). No dataset is included with this tool.</div>

1. Either upload a CSV or type the path of a CSV already on this machine, then select **Check CSV**.
2. Choose a method and grid spacing, optionally a second method to compare with, then select **Reconstruct surface**.

Each result is saved as a CSV and shown beside the input point support. The
CSV also records the distance from every grid node to the nearest measurement,
which the paper uses to report where reconstruction error is largest.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "middleware").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd

import dtin
from dtin import METHODS, describe_load, load_points, make_grid, nearest_distance, prediction_bounds, summarize_model
from dtin.cli import result_table

HEADLESS = os.environ.get("DTIN_NB_INPUT") is not None
try:
    import ipywidgets as widgets
    from IPython.display import FileLink, display
except ImportError:
    widgets = None
    FileLink = None
    HEADLESS = True
try:
    import matplotlib
    if HEADLESS:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

def _writable_dir(*candidates):
    """First candidate directory that can be written to."""
    for cand in candidates:
        try:
            cand.mkdir(parents=True, exist_ok=True)
            probe = cand / ".dtin_write_test"
            probe.write_text("")
            probe.unlink()
            return cand
        except OSError:
            continue
    raise OSError("no writable output directory; set DTIN_NB_OUTPUT_DIR")


OUTPUT_DIR = _writable_dir(
    Path(os.environ.get("DTIN_NB_OUTPUT_DIR", ROOT / "output")),
    Path.home() / "dtin_output",
)

def _result_link(path):
    """Build a browser-usable link for results stored under the notebook root."""
    path = Path(path).resolve()
    try:
        display_path = path.relative_to(ROOT)
    except ValueError:
        display_path = path
    return FileLink(str(display_path), result_html_prefix="Download result: ")


STATE = {"points": None}
RUNS = {}
print(f"dtin {dtin.__version__} from {SRC}; output -> {OUTPUT_DIR}")


## 1. Input

In [ ]:
FORMAT_NOTE = ("CSV columns: lon (degrees), lat (degrees), elev (metres). "
               "Provide the file one way: upload it, or type the path of a CSV already on this machine.")


def load_from(path, min_elev=None):
    pts = load_points(path, min_elev=min_elev)
    STATE["points"] = pts
    print(describe_load(pts))
    return pts


if HEADLESS:
    load_from(os.environ["DTIN_NB_INPUT"])
else:
    format_note = widgets.HTML(f"<div style='color:#5E6B73'>{FORMAT_NOTE}</div>")
    upload = widgets.FileUpload(accept=".csv", multiple=False, description="Upload CSV")
    path_box = widgets.Text(description="File path for large datasets (optional, instead of upload)",
                            placeholder="/path/to/points.csv",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="75%"))
    min_elev_box = widgets.Text(value="", description="Minimum elevation (m)",
                                placeholder="optional", style={"description_width": "initial"})
    check_btn = widgets.Button(description="Check CSV", button_style="info")
    check_out = widgets.Output()

    def _upload_value():
        value = upload.value
        if not value:
            return None
        if isinstance(value, dict):                     # ipywidgets 7
            name, item = next(iter(value.items()))
            return name, bytes(item["content"])
        else:                                           # ipywidgets 8
            item = value[0]
            return item["name"], bytes(item["content"])

    def _uploaded_path():
        item = _upload_value()
        if item is None:
            return None
        name, content = item
        target = OUTPUT_DIR / "uploads" / Path(name).name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
        return target

    def _resolve_path():
        uploaded = _uploaded_path()
        if uploaded is not None:
            return uploaded
        if path_box.value.strip():
            return Path(path_box.value.strip()).expanduser()
        raise ValueError("upload a CSV or type a file path (one of the two is required)")

    def _on_check(_):
        check_out.clear_output()
        with check_out:
            try:
                me = float(min_elev_box.value) if min_elev_box.value.strip() else None
                load_from(_resolve_path(), me)
            except Exception as exc:
                print("Error:", exc)

    check_btn.on_click(_on_check)
    display(widgets.VBox([
        format_note, upload, path_box, min_elev_box,
        check_btn, check_out,
    ]))


## 2. Reconstruct

In [ ]:
import time

MS_SCALES = METHODS["ms-dtin"].params["scales"]
MS_WEIGHTS = METHODS["ms-dtin"].params["weights"]


def reconstruct(method, params, res_m, bbox=None):
    pts = STATE["points"]
    if pts is None:
        raise ValueError("load a file first (step 1)")
    if len(pts) < 3:
        raise ValueError(f"the surface needs at least three non-collinear points, but only {len(pts)} were loaded (check the min-elev filter)")
    xy = pts[["x", "y"]].to_numpy()
    z = pts["elev"].to_numpy()
    label = METHODS[method].label
    spec = make_grid(xy, res_m, bbox=bbox)
    print(f"Step 1/3  fitting {label} on {len(pts)} points ...", flush=True)
    t0 = time.perf_counter()
    model = METHODS[method].make(**params).fit(xy, z)
    print(f"Step 2/3  predicting {spec.shape[1]} x {spec.shape[0]} = {spec.n_cells} grid nodes at {int(res_m)} m ...", flush=True)
    elev = model.predict(spec.nodes, outside="nan")
    seconds = time.perf_counter() - t0
    print("Step 3/3  writing the result CSV ...", flush=True)
    support = nearest_distance(xy, spec.nodes)
    bounds = prediction_bounds(model, elev)
    stamp = time.strftime("%Y%m%d-%H%M%S")
    out = OUTPUT_DIR / f"{method}_{int(res_m)}m_{stamp}.csv"
    n = 1
    while out.exists():                       # two runs within one second
        out = OUTPUT_DIR / f"{method}_{int(res_m)}m_{stamp}-{n}.csv"
        n += 1
    result_table(spec.nodes, elev, support, bounds).to_csv(out, index=False)
    key = f"{label} @ {int(res_m)} m ({out.stem.split('_', 2)[-1]})"
    RUNS[key] = {"label": label, "spec": spec, "elev": elev, "support": support, "bounds": bounds}
    print(summarize_model(model))
    print(f"{spec.shape[1]} x {spec.shape[0]} nodes, {100 * np.isfinite(elev).mean():.1f}% inside the hull, {seconds:.1f} s")
    print(f"distance to nearest measurement: median {np.median(support) / 1000:.2f} km, max {support.max() / 1000:.2f} km")
    if bounds is not None:
        half = 0.5 * (bounds[1] - bounds[0])
        print(f"90% prediction interval half-width: median {np.nanmedian(half):.1f} m")
    print(f"wrote {out}")
    return key, out


def difference_summary(key_a, key_b):
    # Statistics of A minus B over the nodes both surfaces predict.
    a, b = RUNS[key_a], RUNS[key_b]
    diff = a["elev"] - b["elev"]
    ok = np.isfinite(diff)
    if not ok.any():
        return f"{a['label']} minus {b['label']}: no node inside both hulls"
    d = np.abs(diff[ok])
    return (f"{a['label']} minus {b['label']} over {ok.sum()} nodes: "
            f"median |difference| {np.median(d):.1f} m, 95th percentile {np.percentile(d, 95):.1f} m, "
            f"max {d.max():.1f} m")


def show(key_a, key_b=None):
    if plt is None:
        return
    pts = STATE["points"]
    keys = [key_a] + ([key_b] if key_b else [])
    n = 1 + len(keys)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5), squeeze=False)
    ax = axes[0, 0]
    sc = ax.scatter(pts["x"] / 1000, pts["y"] / 1000, c=pts["elev"], s=1, cmap="viridis")
    ax.set_title(f"input ({len(pts)} points)"); ax.set_xlabel("x [km]"); ax.set_ylabel("y [km]"); ax.set_aspect("equal")
    fig.colorbar(sc, ax=ax, label="m")
    for ax, key in zip(axes[0, 1:], keys):
        spec, elev = RUNS[key]["spec"], RUNS[key]["elev"]
        x0, y0, x1, y1 = spec.bbox
        im = ax.imshow(elev.reshape(spec.shape), origin="lower", cmap="viridis",
                       extent=(x0 / 1000, x1 / 1000, y0 / 1000, y1 / 1000), aspect="equal")
        ax.set_title(key); ax.set_xlabel("x [km]")
        fig.colorbar(im, ax=ax, label="m")
    if key_b:
        print(difference_summary(key_a, key_b))
    fig.tight_layout()
    if HEADLESS:
        fig.savefig(OUTPUT_DIR / "last_run.png", dpi=110)
    else:
        plt.show()


if HEADLESS:
    method = os.environ.get("DTIN_NB_METHOD", "dtin")
    compare = os.environ.get("DTIN_NB_COMPARE", "")
    res = float(os.environ.get("DTIN_NB_GRID", "1000"))
    key_a, out_a = reconstruct(method, dict(METHODS[method].params), res)
    key_b = None
    if compare:
        key_b, out_b = reconstruct(compare, dict(METHODS[compare].params), res)
    show(key_a, key_b)
    if FileLink is not None:
        display(_result_link(out_a))
        if key_b:
            display(_result_link(out_b))
else:
    WIDE = {"description_width": "initial"}

    def _weight_controls():
        boxes = [widgets.BoundedFloatText(value=w, min=0.0, max=1.0, step=0.05, description=f"{int(s / 1000)} km",
                                          style=WIDE, layout=widgets.Layout(width="110px"))
                 for s, w in zip(MS_SCALES, MS_WEIGHTS)]
        total = widgets.HTML()
        reset = widgets.Button(description="Paper defaults", layout=widgets.Layout(width="130px"))

        def _update(*_):
            s = sum(b.value for b in boxes)
            ok = abs(s - 1.0) <= 1e-3
            colour = "#2B7A9B" if ok else "#B23A3A"
            total.value = f"<span style='color:{colour}'>&nbsp;sum = {s:.2f}{'' if ok else ' (must be 1)'}</span>"

        def _reset(_):
            for b, w in zip(boxes, MS_WEIGHTS):
                b.value = w

        for b in boxes:
            b.observe(_update, names="value")
        reset.on_click(_reset)
        _update()
        row = widgets.HBox([widgets.Label("MS-D-TIN weights"), *boxes, total, reset])

        def read():
            w = tuple(float(b.value) for b in boxes)
            if abs(sum(w) - 1.0) > 1e-3:
                raise ValueError(f"MS-D-TIN weights must sum to 1 (now {sum(w):.2f})")
            return {"scales": MS_SCALES, "weights": w}
        return [row], read

    def _kriging_controls():
        p = METHODS["kriging"].params
        maxlag = widgets.FloatText(value=p["maxlag"], description="variogram maxlag (m)", style=WIDE)
        support = widgets.FloatText(value=p["support"], description="cell-mean support (m)", style=WIDE)
        variant = widgets.Dropdown(options=["ordinary", "simple"], value=p["variant"], description="variant", style=WIDE)
        note = widgets.HTML("<span style='color:#5E6B73'>&nbsp;kriging is slow; try a coarse grid or a bbox first</span>")
        return [widgets.HBox([maxlag, support, variant]), note], lambda: {
            "maxlag": float(maxlag.value), "support": float(support.value), "variant": variant.value}

    def _method_picker(description, optional):
        options = [(f"{v.label} - {v.help}", k) for k, v in METHODS.items()]
        if optional:
            options = [("none", "")] + options
        dd = widgets.Dropdown(options=options, value="" if optional else "dtin", description=description,
                              style=WIDE, layout=widgets.Layout(width="90%"))
        box = widgets.VBox([])
        state = {"read": lambda: {}}

        def _rebuild(*_):
            if dd.value == "ms-dtin":
                box.children, state["read"] = _weight_controls()
            elif dd.value == "kriging":
                box.children, state["read"] = _kriging_controls()
            else:
                box.children, state["read"] = [], (lambda: {})

        dd.observe(_rebuild, names="value")
        _rebuild()
        return dd, box, state

    method_dd, method_params, method_state = _method_picker("Method", optional=False)
    compare_dd, compare_params, compare_state = _method_picker("Compare with (optional)", optional=True)
    grid_box = widgets.FloatText(value=1000.0, description="grid (m)")
    bbox_box = widgets.Text(value="", description="bbox", placeholder="xmin ymin xmax ymax (metres, optional)",
                            layout=widgets.Layout(width="60%"))
    run_btn = widgets.Button(description="Reconstruct surface", button_style="primary")
    run_out = widgets.Output()

    def _on_run(_):
        run_out.clear_output()
        run_btn.disabled = True
        run_btn.description = "Running ..."
        run_btn.button_style = "warning"
        with run_out:
            try:
                bbox = tuple(float(v) for v in bbox_box.value.split()) if bbox_box.value.strip() else None
                params_a = method_state["read"]()
                params_b = compare_state["read"]() if compare_dd.value else None
                key_a, out_a = reconstruct(method_dd.value, params_a, float(grid_box.value), bbox)
                key_b = None
                if compare_dd.value:
                    print()
                    key_b, out_b = reconstruct(compare_dd.value, params_b, float(grid_box.value), bbox)
                print()
                show(key_a, key_b)
                display(_result_link(out_a))
                if key_b:
                    display(_result_link(out_b))
            except Exception as exc:
                print("Error:", exc)
            finally:
                run_btn.disabled = False
                run_btn.description = "Reconstruct surface"
                run_btn.button_style = "primary"

    run_btn.on_click(_on_run)
    display(widgets.VBox([method_dd, method_params, compare_dd, compare_params,
                          widgets.HBox([grid_box, bbox_box]), run_btn, run_out]))
